In [1]:
# ============================================================
# 0. LIBRERÍAS
# ============================================================
# pip install numpy pandas statsmodels linearmodels

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from linearmodels.panel import PanelOLS

np.random.seed(42)

In [2]:
# ============================================================
# 1. GENERAR DATA SINTÉTICA
# ============================================================

buckets = ["A", "B", "C", "D"]
n_meses = 36

# Elasticidades verdaderas
betas_reales = {
    "A": -3.6,
    "B": -2.1,
    "C": -1.2,
    "D": -0.34,
}

# Efectos fijos de bucket
alpha = {
    "A": 0.12,
    "B": 0.05,
    "C": -0.02,
    "D": -0.10,
}

# Efecto temporal común
gamma = 0.03 * np.sin(
    np.linspace(0, 4 * np.pi, n_meses)
)

registros = []

for i, bucket in enumerate(buckets):

    for mes in range(n_meses):

        # Logaritmo de la tasa
        log_tasa = (
            np.log(0.15)
            + 0.05 * i
            + 0.08 * np.sin(mes / 4 + i * 0.4)
            + np.random.normal(0, 0.025)
        )

        # Logaritmo de la efectividad
        log_efectividad = (
            np.log(0.25)
            + alpha[bucket]
            + gamma[mes]
            + betas_reales[bucket]
              * (log_tasa - np.log(0.15))
            + np.random.normal(0, 0.02)
        )

        registros.append({
            "bucket": bucket,
            "mes": mes,
            "log_tasa": log_tasa,
            "log_efectividad": log_efectividad,
        })

df = pd.DataFrame(registros)

df["tasa_ofertada"] = np.exp(df["log_tasa"])
df["efectividad"] = np.exp(df["log_efectividad"])

print("Dimensiones:", df.shape)
print(df.head())

Dimensiones: (144, 6)
  bucket  mes  log_tasa  log_efectividad  tasa_ofertada  efectividad
0      A    0 -1.884702        -1.313764       0.151874     0.268806
1      A    1 -1.861135        -1.354837       0.155496     0.257989
2      A    2 -1.864620        -1.368240       0.154955     0.254555
3      A    3 -1.803109        -1.562969       0.164786     0.209513
4      A    4 -1.841539        -1.425806       0.158573     0.240315


In [3]:
# ============================================================
# 2. CREAR LAS CUATRO INTERACCIONES
# ============================================================

nombres_interaccion = {}

for bucket in buckets:

    nombre = f"x_{bucket}"

    df[nombre] = (
        df["log_tasa"]
        * (df["bucket"] == bucket).astype(int)
    )

    nombres_interaccion[bucket] = nombre


In [4]:
# ============================================================
# 3. MÉTODO 1: OLS CON DUMMIES E INTERACCIONES
# ============================================================

df["mes_cat"] = df["mes"].astype(str)

formula = (
    "log_efectividad ~ C(bucket) + C(mes_cat) + "
    + " + ".join(nombres_interaccion.values())
)

modelo_ols = smf.ols(
    formula=formula,
    data=df
).fit()

betas_ols = {
    bucket: modelo_ols.params[nombres_interaccion[bucket]]
    for bucket in buckets
}


In [5]:
print(modelo_ols.summary())

                            OLS Regression Results                            
Dep. Variable:        log_efectividad   R-squared:                       0.989
Model:                            OLS   Adj. R-squared:                  0.985
Method:                 Least Squares   F-statistic:                     225.0
Date:                Sat, 24 Oct 2026   Prob (F-statistic):           2.61e-83
Time:                        23:13:55   Log-Likelihood:                 387.37
No. Observations:                 144   AIC:                            -688.7
Df Residuals:                     101   BIC:                            -561.0
Df Model:                          42                                         
Covariance Type:            nonrobust                                         
                       coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------
Intercept           -7.9940      0.143  

In [6]:
# ============================================================
# 4. MÉTODO 2: PanelOLS CON INTERACCIONES
# ============================================================

df_panel = df.set_index(["bucket", "mes"]).sort_index()

columnas_x = list(nombres_interaccion.values())

modelo_panel = PanelOLS(
    dependent=df_panel["log_efectividad"],
    exog=df_panel[columnas_x],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True,
)

resultado_panel = modelo_panel.fit(
    cov_type="unadjusted"
)

betas_panel = {
    bucket: resultado_panel.params[nombres_interaccion[bucket]]
    for bucket in buckets
}


In [7]:
print(resultado_panel)

                          PanelOLS Estimation Summary                           
Dep. Variable:        log_efectividad   R-squared:                        0.9604
Estimator:                   PanelOLS   R-squared (Between):             -11.855
No. Observations:                 144   R-squared (Within):               0.9575
Date:                Sat, Oct 24 2026   R-squared (Overall):             -11.755
Time:                        23:13:55   Log-likelihood                    387.37
Cov. Estimator:            Unadjusted                                           
                                        F-statistic:                      612.33
Entities:                           4   P-value                           0.0000
Avg Obs:                       36.000   Distribution:                   F(4,101)
Min Obs:                       36.000                                           
Max Obs:                       36.000   F-statistic (robust):             612.33
                            

In [8]:
# ============================================================
# 5. MÉTODO 3: DEMEANING GLOBAL DE X E Y
# ============================================================
# Demeaning de dos vías para el panel balanceado:
# variable - media bucket - media mes + media global.
#
# Después estimamos una regresión independiente por bucket.

for variable in ["log_tasa", "log_efectividad"]:

    df[f"{variable}_dm"] = (
        df[variable]
        - df.groupby("bucket")[variable].transform("mean")
        - df.groupby("mes")[variable].transform("mean")
        + df[variable].mean()
    )

betas_separados = {}

resultados = {}

for bucket in buckets:
    d = df.loc[df["bucket"] == bucket]
    
    # Definimos X e Y (sin constante, ya que el demeaning 
    # ya centró las variables en cero)
    y = d["log_efectividad_dm"]
    x = d["log_tasa_dm"]
    
    # Ajustar el modelo OLS
    model = sm.OLS(y, x).fit()
    
    # Guardar los estadísticos clave en el diccionario
    resultados[bucket] = {
        "beta": model.params["log_tasa_dm"],
        "std_err": model.bse["log_tasa_dm"],
        "t_stat": model.tvalues["log_tasa_dm"],
        "p_value": model.pvalues["log_tasa_dm"],
        "R2": model.rsquared,
        "N": model.nobs
    }

# Convertir a DataFrame para ver una tablita ordenada
df_resultados = pd.DataFrame(resultados).T
print(df_resultados)


       beta   std_err    t_stat       p_value        R2     N
A -2.395113  0.455274 -5.260813  7.284512e-06  0.441574  36.0
B -1.678051  0.248297 -6.758246  7.854306e-08  0.566155  36.0
C -1.219695  0.504289 -2.418644  2.091697e-02  0.143204  36.0
D -1.100696  0.328069 -3.355076  1.919744e-03  0.243350  36.0


In [9]:
betas_separados={bucket: beta["beta"] for bucket, beta in resultados.items()}

In [10]:
# ============================================================
# 6. MÉTODO 4: DEMEANING DE CADA INTERACCIÓN
# ============================================================
# Importante:
# transformamos cada interacción sobre TODO el panel.
# Luego estimamos conjuntamente las cuatro pendientes.
#
# No filtramos por bucket para esta regresión final.

columnas_interaccion_dm = []

for bucket in buckets:

    nombre_x = nombres_interaccion[bucket]
    nombre_dm = f"{nombre_x}_dm"

    df[nombre_dm] = (
        df[nombre_x]
        - df.groupby("bucket")[nombre_x].transform("mean")
        - df.groupby("mes")[nombre_x].transform("mean")
        + df[nombre_x].mean()
    )

    columnas_interaccion_dm.append(nombre_dm)

# Regresión sobre las variables transformadas
# Sin intercepto porque las variables ya fueron demeaned.
modelo_demeaning_interacciones = smf.ols(
    "log_efectividad_dm ~ 0 + "
    + " + ".join(columnas_interaccion_dm),
    data=df
).fit()

betas_dm_interacciones = {
    bucket: modelo_demeaning_interacciones.params[
        f"{nombres_interaccion[bucket]}_dm"
    ]
    for bucket in buckets
}


In [11]:

# ============================================================
# 7. COMPARACIÓN FINAL
# ============================================================

comparacion = pd.DataFrame({
    "bucket": buckets,
    "beta_real": [betas_reales[b] for b in buckets],
    "OLS_dummies_interacciones": [
        betas_ols[b] for b in buckets
    ],
    "PanelOLS_interacciones": [
        betas_panel[b] for b in buckets
    ],
    "Separados_demeaning_global_XY": [
        betas_separados[b] for b in buckets
    ],
    "Demeaning_interacciones_conjunto": [
        betas_dm_interacciones[b] for b in buckets
    ],
})

print("\nCOMPARACIÓN DE ELASTICIDADES")
print(comparacion.round(5).to_string(index=False))

print("\nDiferencia máxima OLS vs PanelOLS:")
print(
    np.max([
        abs(betas_ols[b] - betas_panel[b])
        for b in buckets
    ])
)

print("\nDiferencia máxima OLS vs demeaning de interacciones:")
print(
    np.max([
        abs(betas_ols[b] - betas_dm_interacciones[b])
        for b in buckets
    ])
)


COMPARACIÓN DE ELASTICIDADES
bucket  beta_real  OLS_dummies_interacciones  PanelOLS_interacciones  Separados_demeaning_global_XY  Demeaning_interacciones_conjunto
     A      -3.60                   -3.55044                -3.55044                       -2.39511                          -3.55044
     B      -2.10                   -2.10236                -2.10236                       -1.67805                          -2.10236
     C      -1.20                   -1.20115                -1.20115                       -1.21970                          -1.20115
     D      -0.34                   -0.42432                -0.42432                       -1.10070                          -0.42432

Diferencia máxima OLS vs PanelOLS:
1.1102230246251565e-14

Diferencia máxima OLS vs demeaning de interacciones:
1.1102230246251565e-14
